# Genesis to Atoms: Cosmic Nucleosynthesis

**"From the Quark-Gluon Plasma to the First Elements."**

In this simulation, we model the **Hadronization** epoch of the early universe.

**The FTD Atomic Model:**
*   **Fundamental Quanta (±1):** The raw "quarks" or preons of the lattice.
*   **Baryons (Triads):** Stable structure of 3 locked quanta (e.g., Proton).
*   **Nuclei:** Clusters of Baryons fused together.

We will watch the **Primordial Soup** cool and condense into the first stable atomic nuclei.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import label

# Add repository root to path
current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "../../"))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from ternary_matrix.model.grid import Universe
from ternary_matrix.physics import master_equation
from ternary_matrix.config import CONSTANTS

## 1. The Particle Detector

We define a `NucleosynthesisMonitor` to classify structures by their **Baryon Number** (size).

In [ ]:
class NucleosynthesisMonitor:
    def __init__(self, universe):
        self.universe = universe
        
    def scan(self):
        # Detect all matter clusters
        matter_mask = (self.universe.states != 0)
        labels, count = label(matter_mask)
        
        if count == 0:
            return { "preons": 0, "baryons": 0, "nuclei": 0 }
            
        # Get sizes of all structures
        sizes = np.bincount(labels.ravel())[1:]
        
        # Classification Rules
        # Size 1-2: Transient / Preons / Mesons (Unstable)
        # Size 3-5: Baryons (Protons/Neutrons)
        # Size >5:  Atomic Nuclei (Deuterium, Helium, etc)
        
        preons = sum(1 for s in sizes if s < 3)
        baryons = sum(1 for s in sizes if 3 <= s <= 5)
        nuclei = sum(1 for s in sizes if s > 5)
        
        return {
            "preons": preons,
            "baryons": baryons,
            "nuclei": nuclei,
            "max_size": max(sizes) if len(sizes) > 0 else 0
        }
    
    def report(self, t, T_universe):
        stats = self.scan()
        print(f"[T={t}] Temp: {T_universe:.2f} | Preons: {stats['preons']} | Baryons: {stats['baryons']} | Nuclei: {stats['nuclei']} (Max Mass: {stats['max_size']})")
        return stats

## 2. Cosmic Conditions

We simulate "Cooling". 
*   Start with **High flux, High decay** (The Big Bang).
*   Gradually propagate and let decay drop (Expansion & Cooling).

In [ ]:
# Setup
CONSTANTS.C = 0.5
CONSTANTS.ALPHA = 0.02      # Tuned for atomic stability
CONSTANTS.KB = 0.5          # Easy genesis
CONSTANTS.DECAY_RATE = 0.05 # Initial cooling rate
CONSTANTS.GRID_SIZE = 50    

universe = Universe(size=CONSTANTS.GRID_SIZE)
monitor = NucleosynthesisMonitor(universe)

# INFLATION: Massive Injection
universe.flux += np.random.normal(0, 5.0, universe.flux.shape)
print("Big Bang Initiated.")

## 3. Epoch of Recombination (Simulation)

We run for 150 ticks. We expect `Preons` to dominate early, then `Baryons` to form, and finally stable `Nuclei` to emerge as the "Temperature" (Flux density) drops.

In [ ]:
history = {"preons": [], "baryons": [], "nuclei": []}

for t in range(150):
    master_equation.tick(universe)
    
    # Calculate "Universe Temperature" (Average Flux)
    avg_flux = np.mean(np.abs(universe.flux))
    
    if t % 10 == 0:
        stats = monitor.report(t, avg_flux)
        
    # Record Data
    stats = monitor.scan()
    history["preons"].append(stats["preons"])
    history["baryons"].append(stats["baryons"])
    history["nuclei"].append(stats["nuclei"])

print("Nucleosynthesis Complete.")

## 4. The Periodic Table of the Toy Universe

Plotting the abundances over time. Look for the "Baryon Crossover" point where stable matter exceeds unstable preons.

In [ ]:
plt.figure(figsize=(12, 6))
plt.stackplot(range(150), 
              history["preons"],
              history["baryons"], 
              history["nuclei"], 
              labels=['Preon Soup', 'Protons/Neutrons', 'Heavy Nuclei'],
              colors=['#ffcccc', '#ff9999', '#cc0000'])
plt.title("Cosmic Nucleosynthesis History")
plt.xlabel("Time (Epochs)")
plt.ylabel("Population")
plt.legend(loc='upper left')
plt.show()

## 5. Atom Imaging

Visualizing the largest "Atom" (Nucleus) found in the simulation.

In [ ]:
matter = (universe.states != 0)
labels, count = label(matter)

# Find largest cluster
sizes = np.bincount(labels.ravel())
largest_label = sizes[1:].argmax() + 1 # +1 because 0 is background

# Isolate it
atom_mask = (labels == largest_label)
atom_states = np.zeros_like(universe.states)
atom_states[atom_mask] = universe.states[atom_mask]

# Plot
colors = np.empty(atom_states.shape, dtype=object)
colors[atom_states == 1] = 'red'
colors[atom_states == -1] = 'blue'

ax = plt.figure(figsize=(8, 8)).add_subplot(projection='3d')
# Crop view to the atom to see detail
# (finding bounding box)
indices = np.argwhere(atom_mask)
if len(indices) > 0:
    min_z, max_z = indices[:,0].min(), indices[:,0].max()
    min_y, max_y = indices[:,1].min(), indices[:,1].max()
    min_x, max_x = indices[:,2].min(), indices[:,2].max()
    ax.set_xlim(min_x-2, max_x+2)
    ax.set_ylim(min_y-2, max_y+2)
    ax.set_zlim(min_z-2, max_z+2)

ax.voxels(atom_mask, facecolors=colors, edgecolor='k')
ax.set_title(f"Largest Atomic Nucleus (Mass: {sizes[largest_label]}) ")
plt.show()